In [2]:
# ============================================================================
# FIXED IR CAPTION INFERENCE - WITH YOUR PATHS
# ============================================================================

import torch
import torch.nn as nn
import numpy as np
import pandas as pd
import yaml
import json
import re
from pathlib import Path
from collections import Counter

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

class Vocabulary:
    def __init__(self):
        self.word2idx = {'<PAD>': 0, '<SOS>': 1, '<EOS>': 2, '<UNK>': 3}
        self.idx2word = {0: '<PAD>', 1: '<SOS>', 2: '<EOS>', 3: '<UNK>'}
        self.word_count = Counter()
        self.n_words = 4
    
    def tokenize(self, sentence):
        sentence = sentence.lower()
        sentence = re.sub(r'[^\w\s\-\+]', ' ', sentence)
        return sentence.split()
    
    def decode(self, indices):
        words = []
        for idx in indices:
            if idx == self.word2idx['<EOS>']:
                break
            if idx == self.word2idx['<PAD>'] or idx == self.word2idx['<SOS>']:
                continue
            words.append(self.idx2word.get(idx, '<UNK>'))
        return ' '.join(words)

class CaptionModel(nn.Module):
    def __init__(self, feature_dim, vocab_size, embed_dim=256, hidden_dim=512, num_layers=2, dropout=0.3):
        super(CaptionModel, self).__init__()
        self.hidden_dim = hidden_dim
        self.num_layers = num_layers
        
        self.feature_encoder = nn.Sequential(
            nn.Linear(feature_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout)
        )
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.lstm = nn.LSTM(embed_dim + hidden_dim, hidden_dim, num_layers=num_layers, batch_first=True, dropout=dropout if num_layers > 1 else 0)
        self.output = nn.Linear(hidden_dim, vocab_size)
    
    def generate(self, features, vocab, max_len=50, temperature=0.8):
        self.eval()
        if features.dim() == 1:
            features = features.unsqueeze(0)
        
        encoded_features = self.feature_encoder(features)
        current_token = torch.tensor([[vocab.word2idx['<SOS>']]], device=features.device)
        
        generated_indices = []
        hidden = None
        
        for _ in range(max_len):
            embedded = self.embedding(current_token)
            lstm_input = torch.cat([embedded, encoded_features.unsqueeze(1)], dim=2)
            lstm_out, hidden = self.lstm(lstm_input, hidden)
            logits = self.output(lstm_out.squeeze(1))
            logits = logits / temperature
            probs = torch.softmax(logits, dim=-1)
            next_token = torch.multinomial(probs, 1)
            
            if next_token.item() == vocab.word2idx['<EOS>']:
                break
            
            generated_indices.append(next_token.item())
            current_token = next_token
        
        return vocab.decode(generated_indices)

class YOLOFeatureExtractor:
    def __init__(self, yaml_path):
        with open(yaml_path, 'r') as f:
            self.config = yaml.safe_load(f)
        
        self.class_names = self.config['names']
        self.train_path = Path(self.config['train'])
        self.val_path = Path(self.config['val'])
        
        self.train_label_path = self.train_path.parent.parent / 'train' / 'labels'
        self.val_label_path = self.val_path.parent.parent / 'valid' / 'labels'
    
    def get_image_list(self, split='train', only_with_labels=True):
        img_path = self.train_path if split == 'train' else self.val_path
        label_path = self.train_label_path if split == 'train' else self.val_label_path
        
        image_files = []
        for ext in ['.jpg', '.jpeg', '.png', '.bmp', '.JPG', '.JPEG', '.PNG']:
            image_files.extend(list(Path(img_path).glob(f'*{ext}')))
        
        image_files = sorted(image_files)
        
        if only_with_labels:
            image_files = [img for img in image_files 
                          if (label_path / f"{img.stem}.txt").exists()]
            print(f"Found {len(image_files)} images with labels in {split}")
        
        return image_files
    
    def get_label_path(self, image_path, split='train'):
        label_base = self.train_label_path if split == 'train' else self.val_label_path
        return label_base / f"{Path(image_path).stem}.txt"
    
    def parse_yolo_label(self, label_path):
        detections = []
        if not Path(label_path).exists():
            return detections
        
        with open(label_path, 'r') as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) >= 5:
                    detections.append({
                        'class': self.class_names[int(parts[0])],
                        'class_id': int(parts[0]),
                        'x': float(parts[1]),
                        'y': float(parts[2]),
                        'w': float(parts[3]),
                        'h': float(parts[4])
                    })
        return detections
    
    def extract_features(self, detections, max_objects=20):
        person_count = sum(1 for d in detections if d['class'] == 'person')
        large_vehicle_count = sum(1 for d in detections if d['class'] == 'large vehicle')
        small_vehicle_count = sum(1 for d in detections if d['class'] == 'small vehicle')
        
        persons = [d for d in detections if d['class'] == 'person']
        vehicles = [d for d in detections if 'vehicle' in d['class']]
        
        avg_person_x = np.mean([d['x'] for d in persons]) if persons else 0.5
        avg_person_y = np.mean([d['y'] for d in persons]) if persons else 0.5
        avg_vehicle_x = np.mean([d['x'] for d in vehicles]) if vehicles else 0.5
        avg_vehicle_y = np.mean([d['y'] for d in vehicles]) if vehicles else 0.5
        
        x_spread = np.std([d['x'] for d in detections]) if len(detections) > 1 else 0.0
        y_spread = np.std([d['y'] for d in detections]) if len(detections) > 1 else 0.0
        
        global_features = [
            min(person_count, 15) / 15.0,
            min(large_vehicle_count, 15) / 15.0,
            min(small_vehicle_count, 15) / 15.0,
            avg_person_x, avg_person_y, avg_vehicle_x, avg_vehicle_y,
            x_spread, y_spread
        ]
        
        object_features = []
        for i in range(max_objects):
            if i < len(detections):
                d = detections[i]
                class_onehot = [1 if d['class'] == 'person' else 0,
                               1 if d['class'] == 'large vehicle' else 0,
                               1 if d['class'] == 'small vehicle' else 0]
                object_features.extend(class_onehot + [d['x'], d['y'], d['w'], d['h']])
            else:
                object_features.extend([0, 0, 0, 0, 0, 0, 0])
        
        return np.array(global_features + object_features, dtype=np.float32)

class CaptionGenerator:
    def __init__(self, model_path, yaml_path):
        print("Loading model...")
        checkpoint = torch.load(model_path, map_location=device)
        self.vocab = checkpoint['vocab']
        
        self.model = CaptionModel(
            feature_dim=149,
            vocab_size=self.vocab.n_words,
            embed_dim=256,
            hidden_dim=512,
            num_layers=2,
            dropout=0.0
        )
        self.model.load_state_dict(checkpoint['model_state_dict'])
        self.model.to(device)
        self.model.eval()
        
        self.feature_extractor = YOLOFeatureExtractor(yaml_path)
        print(f"✓ Model loaded (vocab size: {self.vocab.n_words})")
    
    def generate_for_image(self, image_path, split='train', temperature=0.8, num_captions=1):
        label_path = self.feature_extractor.get_label_path(image_path, split)
        detections = self.feature_extractor.parse_yolo_label(label_path)
        features = self.feature_extractor.extract_features(detections)
        features_tensor = torch.tensor(features, dtype=torch.float32).to(device)
        
        captions = [self.model.generate(features_tensor, self.vocab, temperature=temperature) 
                   for _ in range(num_captions)]
        
        return {
            'image_path': str(image_path),
            'image_name': Path(image_path).name,
            'captions': captions,
            'num_objects': len(detections),
            'detections': detections
        }
    
    def show_samples(self, split='train', num_samples=10, temperature=0.8):
        image_files = self.feature_extractor.get_image_list(split, only_with_labels=True)[:num_samples]
        
        print(f"\n{'='*80}")
        print(f"GENERATED CAPTIONS - {split.upper()} SPLIT")
        print(f"{'='*80}")
        
        for i, image_path in enumerate(image_files):
            result = self.generate_for_image(image_path, split, temperature, num_captions=3)
            
            print(f"\n{i+1}. {result['image_name']}")
            print(f"   Objects: {result['num_objects']}")
            class_counts = Counter([d['class'] for d in result['detections']])
            print(f"   Classes: {dict(class_counts)}")
            print(f"   Generated captions:")
            for j, caption in enumerate(result['captions']):
                print(f"     {j+1}. {caption}")
    
    def export_all_captions(self, output_file, split='train', temperature=0.8):
        image_files = self.feature_extractor.get_image_list(split, only_with_labels=True)
        
        print(f"\nGenerating captions for {len(image_files)} images...")
        
        results = []
        for i, image_path in enumerate(image_files):
            result = self.generate_for_image(image_path, split, temperature, num_captions=1)
            
            class_counts = Counter([d['class'] for d in result['detections']])
            
            results.append({
                'image_name': result['image_name'],
                'caption': result['captions'][0],
                'num_objects': result['num_objects'],
                'num_persons': class_counts.get('person', 0),
                'num_large_vehicles': class_counts.get('large vehicle', 0),
                'num_small_vehicles': class_counts.get('small vehicle', 0)
            })
            
            if (i+1) % 100 == 0:
                print(f"  Processed {i+1}/{len(image_files)}")
        
        df = pd.DataFrame(results)
        df.to_csv(output_file, index=False)
        print(f"\n✓ Saved {len(results)} captions to {output_file}")
        
        return df

# ============================================================================
# RUN WITH YOUR PATHS
# ============================================================================
print("="*80)
print("IR CAPTION GENERATOR")
print("="*80)

generator = CaptionGenerator(
    model_path='/home/jovyan/AA 25-26/ir_caption_model/caption_model.pt',
    yaml_path='/home/jovyan/AA 25-26/FLIR/data.yaml'
)

# Show 10 samples
generator.show_samples(split='train', num_samples=10, temperature=0.8)

Using device: cuda
IR CAPTION GENERATOR
Loading model...
✓ Model loaded (vocab size: 79)
Found 1354 images with labels in train

GENERATED CAPTIONS - TRAIN SPLIT

1. th_tr_video-PY5tYnNPcGE9WrZuq-frame-002535-dGfdPbs7GoEEyN6wB_jpg.rf.080c58a87715b97f27c2231b46febed8.jpg
   Objects: 13
   Classes: {'person': 3, 'small vehicle': 10}
   Generated captions:
     1. night-time flir image on a residential road 1 person walking on the right sidewalk multiple cars parked along both sides of the road
     2. night-time flir image on a straight road 1 person standing near the right shoulder 6 cars driving and parked along the road
     3. night-time flir image on a residential road 1 person walking on the right sidewalk multiple cars parked along parked along both sides of the road

2. th_tr_video-PY5tYnNPcGE9WrZuq-frame-002550-JwXHRrii8PYBSvQ2R_jpg.rf.da646098d0ac0587120d122ab4de96c2.jpg
   Objects: 11
   Classes: {'person': 2, 'small vehicle': 9}
   Generated captions:
     1. night-time flir 

In [3]:
# Export all captions for train and validation sets
print("="*80)
print("EXPORTING ALL CAPTIONS")
print("="*80)

# Export training set
print("\n[1/2] Exporting TRAIN split...")
train_df = generator.export_all_captions(
    output_file='/home/jovyan/AA 25-26/FLIR/generated_captions_train.csv',
    split='train',
    temperature=0.8
)

# Export validation set
print("\n[2/2] Exporting VAL split...")
val_df = generator.export_all_captions(
    output_file='/home/jovyan/AA 25-26/FLIR/generated_captions_val.csv',
    split='val',
    temperature=0.8
)

# Show summary
print("\n" + "="*80)
print("EXPORT COMPLETE")
print("="*80)
print(f"\nTrain captions: {len(train_df)}")
print(f"Val captions: {len(val_df)}")
print(f"\nFiles saved to:")
print(f"  /home/jovyan/AA 25-26/FLIR/generated_captions_train.csv")
print(f"  /home/jovyan/AA 25-26/FLIR/generated_captions_val.csv")

# Preview some examples
print("\n" + "="*80)
print("SAMPLE CAPTIONS (Train)")
print("="*80)
print(train_df[['image_name', 'caption', 'num_objects']].head(15).to_string())

EXPORTING ALL CAPTIONS

[1/2] Exporting TRAIN split...
Found 1354 images with labels in train

Generating captions for 1354 images...
  Processed 100/1354
  Processed 200/1354
  Processed 300/1354
  Processed 400/1354
  Processed 500/1354
  Processed 600/1354
  Processed 700/1354
  Processed 800/1354
  Processed 900/1354
  Processed 1000/1354
  Processed 1100/1354
  Processed 1200/1354
  Processed 1300/1354

✓ Saved 1354 captions to /home/jovyan/AA 25-26/FLIR/generated_captions_train.csv

[2/2] Exporting VAL split...
Found 1313 images with labels in val

Generating captions for 1313 images...
  Processed 100/1313
  Processed 200/1313
  Processed 300/1313
  Processed 400/1313
  Processed 500/1313
  Processed 600/1313
  Processed 700/1313
  Processed 800/1313
  Processed 900/1313
  Processed 1000/1313
  Processed 1100/1313
  Processed 1200/1313
  Processed 1300/1313

✓ Saved 1313 captions to /home/jovyan/AA 25-26/FLIR/generated_captions_val.csv

EXPORT COMPLETE

Train captions: 1354
Val 

In [9]:
# --- Instead of argparse, set these directly ---
images_dir = "/home/jovyan/AA 25-26/FLIR/train/images"       # <-- CHANGE THIS
labels_dir = "/home/jovyan/AA 25-26/FLIR/train/labels"       # <-- or set to None if no labels
output_file = "flir_grid.png"
cols = 3
rows = 3
cell_size = 480
sample_mode = "random"   # "random" or "first"
seed = random.randint(0, 100)
mixed_view = False       # True = some images without boxes
title = None             # e.g. "FLIR Dataset Samples"

# --- Then call the logic directly ---
from pathlib import Path
import random, cv2, numpy as np

exts = {".jpg", ".jpeg", ".png", ".bmp", ".tiff", ".tif"}
img_dir = Path(images_dir)
all_images = sorted([p for p in img_dir.iterdir() if p.suffix.lower() in exts])

n_needed = cols * rows
random.seed(seed)
if sample_mode == "random":
    selected = random.sample(all_images, min(n_needed, len(all_images)))
else:
    selected = all_images[:n_needed]

processed = []
for i, img_path in enumerate(selected):
    img = cv2.imread(str(img_path))
    if img is None:
        continue
    if labels_dir and not (mixed_view and i % 3 == 0):
        label_path = Path(labels_dir) / (img_path.stem + ".txt")
        h, w = img.shape[:2]
        boxes = parse_yolo_labels(str(label_path), w, h)
        if boxes:
            img = draw_boxes(img, boxes)
    if len(img.shape) == 2:
        img = cv2.cvtColor(img, cv2.COLOR_GRAY2BGR)
    processed.append(img)

sample_h, sample_w = processed[0].shape[:2]
cell_w = cell_size
cell_h = int(cell_w * sample_h / sample_w)

grid = make_grid(processed, cols, rows, cell_w, cell_h, title=title)
cv2.imwrite(output_file, grid)
print(f"Saved: {output_file} ({grid.shape[1]}x{grid.shape[0]})")

Saved: flir_grid.png (1448x1160)


# Caption Model Retraining — Summary

## Problem
91.2% of generated captions contained <UNK> tokens (2,579 total). Root cause: "thermal," "vehicles," and "pedestrians" missing from vocab entirely, plus 39 words appearing only once in 37 training captions.

## What We Changed
1. **Standardized vocabulary**: 127 → 75 unique words. Removed synonyms and one-off words.
2. **Data augmentation**: 37 → 74 training captions by paraphrasing existing image descriptions. No new images needed.
3. **Training improvements**: Added dropout (0.3), teacher forcing, gradient clipping, LR scheduling.

## Results

| Metric                 | Before | After  | Change  |
|------------------------|--------|--------|---------|
| Captions with UNK      | 91.2%  | 0.0%   | -91.2%  |
| UNK Token Rate         | 9.6%   | 0.0%   | -9.6%   |
| Valid Caption Opener    | 38.0%  | 99.9%  | +61.9%  |
| Detection Consistency   | 70.5%  | 82.5%  | +12.0%  |
| Caption Diversity       | 68.4%  | 15.4%  | -53.0%  |

## Next Step
Expand training captions to 200+ to improve diversity while maintaining 0% UNK rate.

- Is more robust vocab really better?
- clear and concise is better? 
- the more direct the descriptions are, the better it could be
- in terms of "hw many ways can it tell me how a car is driving down the road" its better that its less robust
- does not need model to be a lyrical genius. 
- if for FLIR imagery, do we really need to be saying "flir image ..." does not need that

- how to get quantitative result for the "correctness" of the model?
- image classifier works by doing confidence intervals, maybe a way that when the LLM uses the classifier it basically is describing the image 
- take the confidence of the model classifying an action and use that as a perspective on model output
- if we can create our desired mapping from telling the model here are the truth labels and this is your training info and this is what youll make predictions on in the future as long as we can have an input and get into an output we will be good. 